# 11. Cross-Attention + Xử lý mất cân bằng lớp

Notebook tích hợp các kỹ thuật xử lý mất cân bằng tốt nhất từ notebook `09`/`10` vào kiến trúc **Cross-Attention** (Phase 5). Tất cả config đóng băng cả hai encoder (PhoBERT + CLIP), chỉ train khối cross-attention và head — giống `fusion_cross_attn_frozen` baseline.

| ID | Config | Mô tả |
|---|---|---|
| CA-base | `fusion_cross_attn_frozen` | Baseline cross-attention (CE thuần) |
| CA-LA | `fusion_cross_attn_frozen_logit_adjustment` | + Logit Adjustment τ=1.0 |
| CA-MX | `fusion_cross_attn_frozen_mixup` | + Minority-aware Manifold Mixup α=0.2 |
| CA-LAMX | `fusion_cross_attn_frozen_la_mixup` | + LA + Mixup kết hợp |

**Hyperparameter chống overfitting so với baseline:**
- `lr_head`: 5e-4 (giảm từ 1e-3 mặc định)
- `weight_decay`: 0.05 (tăng từ 0.01)
- `dropout`: 0.3 (tăng từ 0.2)
- `early_stop_patience`: 5 (tăng để Logit Adjustment có đủ epoch ổn định)

**Post-hoc calibration (B8):** Sau khi train, học hệ số `α_c` tối ưu Macro-F1 trên OOF/val, áp lên test, in bảng so sánh trước/sau.

**Thước đo chính:** Macro-F1 (4 lớp). Val = public test (1.413 mẫu), test = private test (1.504 mẫu).

**Kaggle Input cần Add:**
1. `phuclkz/vimmsd-private-dataset` — ảnh và nhãn gốc (bắt buộc)
2. `vtnmai/vimmsd-data-preprocessed` — caption đã tách từ
3. *(Khuyến nghị)* Output notebook `05` hoặc `11` phiên bản trước → đặt vào `PREVIOUS_OUTPUT`

**Settings:** GPU T4 x2, **Internet ON**.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/meo225/Multimodal_Sarcasm_Detection.git"
REF  = "main"              # branch hoặc commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"   # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
RUN_TESTS = True
if RUN_TESTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
    result = subprocess.run(
        [sys.executable, "-m", "pytest", "tests", "-q", "-x", "-p", "no:cacheprovider"],
        capture_output=True, text=True
    )
    print(result.stdout[-5000:], result.stderr[-2000:])
    if result.returncode != 0:
        raise RuntimeError("pytest fail — xem output phía trên trước khi train")

In [ ]:
import gc
import json
import logging
import re
import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

from vimmsd.utils.config import load_config

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)
pd.set_option("display.max_colwidth", 120)

# ─── CẤU HÌNH CHÍNH ───────────────────────────────────────────────────────────
# Danh sách config sẽ chạy trong notebook này.
# Có thể bỏ bớt config hoặc seed để chạy nhanh hơn.
RUNS = [
    "fusion_cross_attn_frozen",           # CA-base: baseline (CE thuần)
    "fusion_cross_attn_frozen_logit_adjustment",  # CA-LA
    "fusion_cross_attn_frozen_mixup",             # CA-MX
    "fusion_cross_attn_frozen_la_mixup",          # CA-LAMX
]
SEEDS = [42, 1, 2]   # 42 = seed đồng nhất với các phase trước
SPLIT_SEED = 42      # cố định split train/val/test
SKIP_DONE = True     # True: bỏ qua run đã có test_metrics.json
PREVIOUS_OUTPUT = None  # Đặt đường dẫn output version trước nếu muốn tái sử dụng (vd "/kaggle/input/<tên>/outputs")
RARE = "text-sarcasm"   # lớp siêu hiếm (7 mẫu test), đọc macro-F1 cẩn thận

# MODES mô tả ngắn từng config để hiển thị bảng
MODES = {
    "fusion_cross_attn_frozen":                   "CA baseline (CE)",
    "fusion_cross_attn_frozen_logit_adjustment":   "CA + Logit Adj. τ=1.0",
    "fusion_cross_attn_frozen_mixup":              "CA + Manifold Mixup α=0.2",
    "fusion_cross_attn_frozen_la_mixup":           "CA + LA + Mixup",
}


def run_name(name, seed):
    return f"{name}_s{seed}"


cfgs = {name: load_config(f"configs/{name}.yaml") for name in RUNS}
OUT = Path(next(iter(cfgs.values())).paths.output_dir)
LABELS = list(next(iter(cfgs.values())).data.labels)
print("output:", OUT)
print("labels:", LABELS)
print({name: (cfg.train.epochs, cfg.train.lr_head) for name, cfg in cfgs.items()})

## 1. Kiểm tra dữ liệu

Báo lỗi sớm nếu thiếu dataset, thay vì đợi tới lúc train.

In [ ]:
from vimmsd.data.dataset import load_all_records, stage_preprocessed_cache

cfg0 = next(iter(cfgs.values()))
data_dir = Path(cfg0.paths.data_dir)
if not (data_dir / cfg0.data.train_json).exists():
    raise FileNotFoundError(
        f"Không thấy {data_dir / cfg0.data.train_json}: "
        "Add Input phuclkz/vimmsd-private-dataset"
    )

for path in stage_preprocessed_cache(cfg0):
    meta = json.loads(path.read_text(encoding="utf-8")).get("meta", {})
    print(f"{path.name}: pipeline_key={meta.get('pipeline_key')}, "
          f"preprocess_version={meta.get('preprocess_version')}")

splits = load_all_records(cfg0)
counts = pd.DataFrame(
    {split: Counter(LABELS[r["label"]] for r in splits[split])
     for split in ("train", "val", "test")}
).reindex(LABELS)
counts["train (%)"] = (100 * counts["train"] / counts["train"].sum()).round(2)
display(counts)

assert (len(splits["train"]), len(splits["val"]), len(splits["test"])) == (10805, 1413, 1504), \
    "Số mẫu không khớp split chính thức"

# Hiển thị mức độ mất cân bằng
fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(LABELS, counts["train"], color="#4c72b0")
ax.set_xscale("log")
for y, v in enumerate(counts["train"]):
    ax.text(v, y, f" {v}", va="center")
ax.set_title("Phân bố nhãn train (thang log) — tỉ lệ 6062:77:442:4224")
plt.tight_layout()
plt.show()

## 2. Giải thích các kỹ thuật xử lý mất cân bằng

### 2.1 Logit Adjustment (LA) — τ = 1.0

Với prior $\pi_c = N_c / \sum N$, LA cộng độ dịch tần suất vào logit **chỉ trong lúc train**:

$$L_{\text{LA}} = \text{CE}\bigl(f(x) + \tau\log\pi,\; y\bigr)$$

Mô hình buộc phải học logit thô lớn hơn cho lớp hiếm để bù lại $\tau\cdot\log(\pi_c)$ âm mạnh. **Khi inference, dùng logit thô** — không cộng lại độ dịch.

### 2.2 Minority-aware Manifold Mixup — α = 0.2

Với vector fusion $z$ sau cross-attention blocks, chọn hoán vị $j$ trong mini-batch và trộn:

$$\tilde{z} = \lambda z_i + (1-\lambda)z_j, \qquad \tilde{y} = \lambda y_i + (1-\lambda)y_j, \qquad \lambda \sim \text{Beta}(\alpha, \alpha)$$

Head MLP nhận $\tilde{z}$, loss dùng **soft target** $\tilde{y}$. Nếu anchor $i$ là `text-sarcasm` hoặc `image-sarcasm` và $\lambda < 0.5$, đổi $\lambda \leftarrow 1-\lambda$: anchor hiếm giữ ít nhất 50% đặc trưng.

### 2.3 Post-hoc Class-wise Calibration

Giữ nguyên checkpoint, tìm $\alpha = (\alpha_0,\ldots,\alpha_3)$ tối ưu Macro-F1:

$$\hat{y} = \arg\max_c \; \alpha_c \cdot P(y=c|x)$$

$\alpha_0 = 1$ (lớp tham chiếu). Tìm 3 hệ số còn lại trên log-scale bằng `scipy.optimize.minimize` / `differential_evolution` **trên val**. Áp lên test và in bảng so sánh.

In [ ]:
# Minh hoạ Logit Adjustment: hiển thị τ·log(prior) của từng lớp
from vimmsd.training.losses import LogitAdjustmentLoss

class_counts = [int(counts.loc[label, "train"]) for label in LABELS]
priors = np.array(class_counts) / np.sum(class_counts)
tau = 1.0
demo_df = pd.DataFrame({
    "lớp": LABELS,
    "N_c": class_counts,
    "prior π_c": priors.round(4),
    "τ·log(π_c) (độ dịch âm)": (tau * np.log(priors)).round(4),
})
display(demo_df)
print("→ text-sarcasm bị phạt nặng nhất; mô hình phải học logit thô cao hơn để bù.")

# Kiểm tra implementation khớp công thức thủ công
torch.manual_seed(42)
demo_logits = torch.randn(4, 4)
demo_y = torch.arange(4)
b6_loss = LogitAdjustmentLoss(class_counts, tau=tau)
manual = F.cross_entropy(
    demo_logits + tau * torch.log(torch.tensor(priors, dtype=demo_logits.dtype)), demo_y
)
print(f"Loss (implementation): {float(b6_loss(demo_logits, demo_y)):.6f}")
print(f"Loss (công thức thủ công): {float(manual):.6f}")
assert torch.isclose(b6_loss(demo_logits, demo_y), manual), "❌ Implementation không khớp!"
print("✓ Implementation khớp công thức.")

In [ ]:
# Minh hoạ Manifold Mixup: kiểm tra anchor hiếm giữ ≥50% đặc trưng
from vimmsd.models.fusion import manifold_mixup

mixup_cfg = cfgs["fusion_cross_attn_frozen_mixup"]
minority_ids = tuple(LABELS.index(x) for x in mixup_cfg.loss.mixup_minority_labels)
print(f"Minority label IDs: {minority_ids} → {[LABELS[i] for i in minority_ids]}")

torch.manual_seed(7)
z_demo = torch.randn(8, 128)  # 128-dim fusion vector
y_demo = torch.tensor([0, 1, 0, 2, 3, 0, 2, 3])  # 1=text-sarcasm, 2=image-sarcasm
z_mix, lam, perm = manifold_mixup(z_demo, y_demo, alpha=0.2, minority_labels=minority_ids)

demo_mix = pd.DataFrame({
    "anchor label": [LABELS[i] for i in y_demo.tolist()],
    "partner label": [LABELS[i] for i in y_demo[perm].tolist()],
    "lambda (anchor weight)": lam.squeeze(1).numpy().round(3),
})
demo_mix["anchor hiếm?"] = y_demo.numpy().tolist()
demo_mix["anchor hiếm?"] = demo_mix["anchor hiếm?"].apply(lambda x: "✓" if x in minority_ids else "")
display(demo_mix)

assert (lam[torch.tensor([i in minority_ids for i in y_demo.tolist()])] >= 0.5).all(), \
    "❌ Anchor hiếm không được giữ ≥50%!"
assert torch.allclose(z_mix, lam * z_demo + (1 - lam) * z_demo[perm])
print("✓ Manifold Mixup hoạt động đúng.")

## 3. Số parameters train được

Build từng model trên CPU để đếm, rồi giải phóng. Số lượng param giống nhau giữa các config (chỉ loss khác nhau).

In [ ]:
from vimmsd.models.classifier import build_model

param_rows = []
for name, cfg in cfgs.items():
    model = build_model(cfg)
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    head = sum(
        p.numel() for n, p in model.named_parameters()
        if p.requires_grad and not n.startswith(("text_encoder.", "image_encoder."))
    )
    param_rows.append({
        "config": name, "trainable (M)": trainable / 1e6,
        "trong đó encoder (M)": (trainable - head) / 1e6,
        "head+fusion (M)": head / 1e6, "tổng (M)": total / 1e6
    })
    del model
    gc.collect()

params_df = pd.DataFrame(param_rows).set_index("config")
params_df.round(2)

## 4. Train + Đánh giá

Mỗi config × seed: `train.py` (train, chọn best checkpoint theo val macro-F1, chấm test) rồi `evaluate.py --split test`.

**2 GPU:** mỗi run khoảng 1–2 GB VRAM, chạy 2 run song song (`vimmsd.utils.gpu_queue`).

Session bị ngắt: đặt `PREVIOUS_OUTPUT` trỏ vào output version trước rồi chạy lại cell này.

In [ ]:
import shutil
import torch
from vimmsd.utils.gpu_queue import Job, run_on_gpus

NUM_GPUS = torch.cuda.device_count()
print("GPU:", NUM_GPUS, [torch.cuda.get_device_name(k) for k in range(NUM_GPUS)])

# ─── Tái sử dụng output từ phiên trước ──────────────────────────────────────
if PREVIOUS_OUTPUT and Path(PREVIOUS_OUTPUT).resolve() == OUT.resolve():
    print("PREVIOUS_OUTPUT trùng OUT: kết quả đã nằm sẵn ở đây, không cần copy")
elif PREVIOUS_OUTPUT:
    if not Path(PREVIOUS_OUTPUT).is_dir():
        raise FileNotFoundError(f"Không thấy PREVIOUS_OUTPUT={PREVIOUS_OUTPUT}")
    for metrics in sorted(Path(PREVIOUS_OUTPUT).glob("*/test_metrics.json")):
        shutil.copytree(
            metrics.parent, OUT / metrics.parent.name,
            dirs_exist_ok=True, ignore=shutil.ignore_patterns("*.pt")
        )
        print("dùng lại", metrics.parent.name)


def make_job(name, seed):
    run = run_name(name, seed)
    out_dir = OUT / run
    overrides = [f"seed={seed}", f"data.split_seed={SPLIT_SEED}", f"name={run}"]
    train_cmd = [
        sys.executable, "scripts/train.py",
        "--config", f"configs/{name}.yaml",
        "--override", *overrides,
        "--note", f"Phase 11 {name} seed {seed}",
    ]
    eval_cmd = [
        sys.executable, "scripts/evaluate.py",
        "--config", f"configs/{name}.yaml",
        "--split", "test",
        "--override", *overrides,
    ]
    # Thêm --split val để lưu xác suất cho post-hoc calibration
    eval_val_cmd = [
        sys.executable, "scripts/evaluate.py",
        "--config", f"configs/{name}.yaml",
        "--split", "val",
        "--override", *overrides,
    ]

    def check():
        missing = [
            f for f in ("test_metrics.json", "test_predictions.csv")
            if not (out_dir / f).exists()
        ]
        if missing:
            return f"thiếu {', '.join(missing)}"
        (out_dir / "last.pt").unlink(missing_ok=True)
        return None

    return Job(run, [train_cmd, eval_cmd, eval_val_cmd], out_dir / "run.log", check=check)


jobs = []
for seed in SEEDS:
    for name in RUNS:
        out_dir = OUT / run_name(name, seed)
        if SKIP_DONE and (out_dir / "test_metrics.json").exists() and (out_dir / "test_predictions.csv").exists():
            print(f"=== bỏ qua {run_name(name, seed)}: đã có kết quả")
            for f in [out_dir / "last.pt", *out_dir.glob("*.tmp")]:
                f.unlink(missing_ok=True)
            continue
        shutil.rmtree(out_dir, ignore_errors=True)
        jobs.append(make_job(name, seed))

print(f"=== {len(jobs)} run, {max(NUM_GPUS, 1)} run cùng lúc")
OUT.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUT).free / 1e9
print(f"Đĩa còn trống: {free_gb:.1f} GB")
if jobs and free_gb < 1:
    raise RuntimeError("Đĩa gần đầy: xóa bớt file trước khi train")

finished = run_on_gpus(jobs, NUM_GPUS, summary_pattern=r"INFO epoch|early stopping|^\| \d{4}-")
print(f"Đĩa còn trống: {shutil.disk_usage(OUT).free / 1e9:.1f} GB")
failed = [job for job in finished if job.error]
if failed:
    raise RuntimeError("run lỗi: " + ", ".join(f"{job.name} ({job.error})" for job in failed))

## 5. Bảng so sánh kết quả

- **val**: best val macro-F1 (checkpoint được chọn)
- **test**: trên private test
- **macro-F1 (3 lớp)**: bỏ `text-sarcasm` (7 mẫu test, nhiễu), ít nhiễu hơn
- Đọc F1 từng lớp cẩn thận: tăng recall lớp hiếm mà precision/not-sarcasm sụt mạnh là đánh đổi xấu

In [ ]:
def parse_results_row(path):
    if not path.exists():
        return {}
    line = path.read_text(encoding="utf-8").strip().splitlines()[-1]
    cols = [c.strip() for c in line.strip("|").split("|")]
    vram = re.search(r"peak VRAM ([\d.]+) GB", line)
    return {
        "commit": cols[2].strip("`"),
        "train (phút)": float(cols[6]),
        "peak VRAM (GB)": float(vram.group(1)) if vram else None,
    }


def metrics_row(config, seed, out_dir, mode):
    test = json.loads((out_dir / "test_metrics.json").read_text())
    val_path = out_dir / "best_val_metrics.json"
    per_class = test["per_class"]
    row = {
        "config": config, "seed": seed, "chế độ": mode,
        "val macro-F1": json.loads(val_path.read_text())["macro_f1"] if val_path.exists() else np.nan,
        "test macro-F1": test["macro_f1"],
        "test macro-F1 (3 lớp)": np.mean([m["f1"] for c, m in per_class.items() if c != RARE]),
        "test weighted-F1": test["weighted_f1"],
        "test macro-P": test["macro_precision"],
        "test macro-R": test["macro_recall"],
        "accuracy": test["accuracy"],
    }
    row.update({f"F1 {c}": m["f1"] for c, m in per_class.items()})
    row.update(parse_results_row(out_dir / "results_row.md"))
    return row


rows = [
    metrics_row(name, seed, OUT / run_name(name, seed), MODES[name])
    for seed in SEEDS
    for name in RUNS
    if (OUT / run_name(name, seed) / "test_metrics.json").exists()
]

runs_df = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
print("Chi tiết từng run:")
display(runs_df.set_index(["config", "seed"]).round(4))

In [ ]:
METRICS = [
    "val macro-F1", "test macro-F1", "test macro-F1 (3 lớp)",
    "test weighted-F1", "accuracy", *[f"F1 {c}" for c in LABELS]
]
RESOURCES = ["train (phút)", "peak VRAM (GB)"]
order = [c for c in RUNS if c in set(runs_df["config"])]
grouped = runs_df.groupby("config", sort=False)
mean = grouped[METRICS + RESOURCES].mean().reindex(order)
std = grouped[METRICS].std().reindex(order)
n_seeds = grouped.size().reindex(order)

summary = pd.DataFrame(index=order)
summary["chế độ"] = grouped["chế độ"].first().reindex(order)
summary["số seed"] = n_seeds
for col in METRICS:
    summary[col] = [
        f"{m:.3f}" if np.isnan(s) else f"{m:.3f} ± {s:.3f}"
        for m, s in zip(mean[col], std[col])
    ]
for col in RESOURCES:
    summary[col] = mean[col].round(2)
summary = summary.join(params_df[["trainable (M)"]].round(2), how="left")

# Tính delta so với baseline
if "fusion_cross_attn_frozen" in mean.index:
    baseline_f1 = mean.loc["fusion_cross_attn_frozen", "test macro-F1"]
    summary["Δ test macro-F1 vs CA-base"] = [
        f"{mean.loc[c, 'test macro-F1'] - baseline_f1:+.3f}" if c in mean.index else ""
        for c in order
    ]

print("Trung bình ± độ lệch chuẩn qua các seed:")
display(summary)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
x = np.arange(len(order))
for offset, col in ((-0.2, "test macro-F1"), (0.2, "test macro-F1 (3 lớp)")):
    axes[0].bar(x + offset, mean[col], 0.4, yerr=std[col].fillna(0), capsize=3, label=col)
axes[0].set_xticks(x, [MODES.get(c, c) for c in order], rotation=20, ha="right", fontsize=8)
axes[0].set_ylabel("F1")
axes[0].set_title("Macro-F1 trên test (trung bình ± std qua seed)")
axes[0].legend(fontsize=8)

width = 0.8 / len(order)
for k, config in enumerate(order):
    cols = [f"F1 {c}" for c in LABELS]
    axes[1].bar(
        np.arange(len(LABELS)) + (k - (len(order) - 1) / 2) * width,
        mean.loc[config, cols], width,
        yerr=std.loc[config, cols].fillna(0), capsize=2,
        label=MODES.get(config, config)
    )
axes[1].set_xticks(np.arange(len(LABELS)), LABELS)
axes[1].set_ylabel("F1")
axes[1].set_title("F1 từng lớp")
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.savefig("reports/figures/phase11_comparison.png", dpi=150)
plt.show()

## 6. Diễn biến train

Đọc từ `run.log` (seed đầu tiên). Cần thấy:
- Train loss giảm đều, không NaN
- Val macro-F1 đạt đỉnh rồi dừng (early stopping)
- CA-LA/CA-MX không overfit nhanh hơn CA-base

In [ ]:
EPOCH_RE = re.compile(
    r"epoch (\d+) \| train loss ([\d.]+) \| val chọn checkpoint \([^)]*\) ([\d.]+) "
    r"\| val macro-F1 ([\d.]+) \| macro-P ([\d.]+) \| macro-R ([\d.]+) \| acc ([\d.]+)"
)


def read_history(name, seed):
    log = OUT / run_name(name, seed) / "run.log"
    if not log.exists():
        return pd.DataFrame()
    rows = [
        dict(zip(
            ["epoch", "train loss", "val select", "val macro-F1",
             "val macro-P", "val macro-R", "val acc"],
            map(float, m.groups())
        ))
        for m in map(EPOCH_RE.search, log.read_text(encoding="utf-8").splitlines()) if m
    ]
    return pd.DataFrame(rows)


histories = {name: read_history(name, SEEDS[0]) for name in RUNS}
histories = {name: h for name, h in histories.items() if len(h)}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for name, h in histories.items():
    label = MODES.get(name, name)
    axes[0].plot(h["epoch"], h["train loss"], marker="o", label=label)
    axes[1].plot(h["epoch"], h["val macro-F1"], marker="o", label=label)
    axes[2].plot(h["epoch"], h["val acc"], marker="o", label=label)
for ax, title in zip(axes, ("train loss", "val macro-F1", "val accuracy")):
    ax.set_title(f"{title} (seed {SEEDS[0]})")
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
axes[1].legend(fontsize=7)
plt.tight_layout()
plt.show()

if histories:
    best_epochs = pd.DataFrame({
        name: {
            "epoch tốt nhất": int(h.loc[h["val macro-F1"].idxmax(), "epoch"]),
            "số epoch chạy": int(h["epoch"].max()),
            "val macro-F1 tốt nhất": h["val macro-F1"].max(),
            "train loss cuối": h["train loss"].iloc[-1],
        }
        for name, h in histories.items()
    }).T
    display(best_epochs.round(4))

## 7. Post-hoc Class-wise Calibration

Áp dụng calibration lên kết quả của config tốt nhất (mặc định `CA-LA`). Dùng **val** để tối ưu `α`, sau đó áp lên test.

**Chống leakage:**
1. Tối ưu `α` trên val (1413 mẫu)
2. Báo cáo kết quả test trước/sau calibration — test **không** tham gia tối ưu
3. Không report điểm val-fitting như bằng chứng tổng quát hóa

> **Lưu ý:** Với tập val nhỏ (1 mẫu `text-sarcasm`, 28 mẫu `image-sarcasm`), calibration không ổn định cho lớp hiếm. Nếu `α` cho text/image quá lớn hoặc val giảm, giữ nguyên kết quả train.

In [ ]:
from vimmsd.analysis.calibration import apply_class_multipliers, tune_class_multipliers
from vimmsd.training.metrics import compute_metrics

prob_cols = [f"prob_{label}" for label in LABELS]


def load_predictions(path):
    df = pd.read_csv(path)
    y = df["label"].map({label: i for i, label in enumerate(LABELS)}).to_numpy(dtype=int)
    p = df[prob_cols].to_numpy(dtype=float)
    assert np.allclose(p.sum(axis=1), 1.0, atol=1e-3), "Xác suất không tổng bằng 1!"
    return df, p, y


# Chọn config tốt nhất từ bảng mean
best_config = max(
    [c for c in order if c in mean.index],
    key=lambda c: mean.loc[c, "test macro-F1"],
    default="fusion_cross_attn_frozen_logit_adjustment"
)
CALIB_RUN = run_name(best_config, SEEDS[0])
print(f"Config tốt nhất: {best_config} → calibrate run {CALIB_RUN}")

val_pred_path = OUT / CALIB_RUN / "val_predictions.csv"
test_pred_path = OUT / CALIB_RUN / "test_predictions.csv"

if not val_pred_path.exists():
    print(f"⚠ Chưa có {val_pred_path}. Chạy evaluate.py --split val thủ công nếu cần.")
elif not test_pred_path.exists():
    print(f"⚠ Chưa có {test_pred_path}.")
else:
    val_df, val_p, val_y = load_predictions(val_pred_path)
    test_df, test_p, test_y = load_predictions(test_pred_path)
    print(f"Val: {val_p.shape}, label dist: {Counter(val_y)}")
    print(f"Test: {test_p.shape}, label dist: {Counter(test_y)}")

    # ─── Tối ưu α trên val ──────────────────────────────────────────────────
    alpha, fit_info = tune_class_multipliers(
        val_p, val_y, bounds=(0.1, 10.0), seed=42, maxiter=100, popsize=15
    )
    alpha_df = pd.DataFrame({"lớp": LABELS, "alpha": alpha})
    alpha_df["ý nghĩa"] = alpha_df["alpha"].apply(
        lambda a: "tăng confidence" if a > 1.1 else "giảm confidence" if a < 0.9 else "giữ nguyên"
    )
    print("\n--- Hệ số calibration học được (trên val) ---")
    display(alpha_df)
    display(pd.Series(fit_info, name="Thông tin fit"))
    assert alpha[0] == 1.0, "α_not-sarcasm phải là 1.0 (lớp tham chiếu)"

    # ─── Áp lên test ────────────────────────────────────────────────────────
    calibrated_test_p = apply_class_multipliers(test_p, alpha, normalize=True)
    before_test = compute_metrics(test_y, test_p.argmax(axis=1), LABELS)
    after_test = compute_metrics(test_y, calibrated_test_p.argmax(axis=1), LABELS)

    def _make_row(mid, desc, m):
        return {
            "ID": mid, "kỹ thuật": desc,
            "macro-F1": m["macro_f1"],
            "macro-F1 bỏ text": np.mean([v["f1"] for c, v in m["per_class"].items() if c != RARE]),
            "weighted-F1": m["weighted_f1"], "accuracy": m["accuracy"],
        }

    calib_table = pd.DataFrame([
        _make_row("Trước", f"{MODES[best_config]} (test)", before_test),
        _make_row("Sau", f"{MODES[best_config]} + Calibration (test)", after_test),
    ]).set_index("ID")
    calib_table["Δ macro-F1"] = calib_table["macro-F1"] - calib_table.loc["Trước", "macro-F1"]
    print("\n--- Kết quả test trước/sau calibration ---")
    display(calib_table.style.format(precision=4))

    # F1 từng lớp
    class_rows = []
    for mid, m in (("Trước", before_test), ("Sau", after_test)):
        for label, v in m["per_class"].items():
            class_rows.append({"ID": mid, "lớp": label, "precision": v["precision"],
                                "recall": v["recall"], "f1": v["f1"]})
    class_df = pd.DataFrame(class_rows)
    display(
        class_df.pivot(index="ID", columns="lớp", values=["precision", "recall", "f1"]).round(4)
    )

    # ─── Lưu kết quả calibration ─────────────────────────────────────────────
    calib_dir = OUT / f"{CALIB_RUN}_calibrated"
    calib_dir.mkdir(parents=True, exist_ok=True)
    (calib_dir / "calibration.json").write_text(
        json.dumps({"labels": LABELS, "alpha": alpha.tolist(), "fit_val": fit_info}, ensure_ascii=False, indent=2)
    )
    (calib_dir / "test_metrics.json").write_text(json.dumps(after_test, indent=2))
    saved = test_df.copy()
    saved["pred"] = [LABELS[i] for i in calibrated_test_p.argmax(axis=1)]
    saved[prob_cols] = calibrated_test_p
    saved.to_csv(calib_dir / "test_predictions.csv", index=False)
    print(f"\n✓ Đã lưu kết quả calibration tại {calib_dir}")

    # ─── Biểu đồ trước/sau ───────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for ax, metric in zip(axes, ("f1", "recall")):
        class_df.pivot(index="lớp", columns="ID", values=metric).reindex(LABELS).plot.bar(ax=ax, rot=0)
        ax.set_title(f"{metric} trước/sau calibration (test)"); ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

## 8. Ma trận nhầm lẫn (test, seed đầu tiên)

In [ ]:
done = [name for name in RUNS if (OUT / run_name(name, SEEDS[0]) / "confusion_matrix_test.png").exists()]
if done:
    fig, axes = plt.subplots(1, len(done), figsize=(5 * len(done), 4.5), squeeze=False)
    for ax, name in zip(axes[0], done):
        ax.imshow(plt.imread(OUT / run_name(name, SEEDS[0]) / "confusion_matrix_test.png"))
        ax.set_title(f"{MODES[name]}\n(seed {SEEDS[0]})", fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("Chưa có confusion matrix — chạy cell train trước.")

## 9. So sánh từng mẫu: CA-LA vs CA-base

Đếm mẫu CA-LA đúng mà CA-base sai ("giúp") và ngược lại ("nhiễu"), theo từng nhãn thật.

In [ ]:
HELP = "giúp (CA-LA đúng, baseline sai)"
HURT = "nhiễu (baseline đúng, CA-LA sai)"

base_name = "fusion_cross_attn_frozen"
la_name = "fusion_cross_attn_frozen_logit_adjustment"

if all(
    (OUT / run_name(n, SEEDS[0]) / "test_predictions.csv").exists()
    for n in (base_name, la_name)
):
    base_pred = pd.read_csv(OUT / run_name(base_name, SEEDS[0]) / "test_predictions.csv").set_index("id")
    la_pred   = pd.read_csv(OUT / run_name(la_name,   SEEDS[0]) / "test_predictions.csv").set_index("id")

    both = base_pred[["label", "pred", "caption"]].join(la_pred[["pred"]], rsuffix="_la")
    base_ok = both["pred"] == both["label"]
    la_ok   = both["pred_la"] == both["label"]
    both[HELP] = la_ok & ~base_ok
    both[HURT] = base_ok & ~la_ok

    summary_cmp = both.groupby("label")[[HELP, HURT]].sum()
    summary_cmp.insert(0, "n", both.groupby("label").size())
    summary_cmp.loc["tổng"] = summary_cmp.sum()
    print(f"=== CA-base vs CA-LA (seed {SEEDS[0]})")
    display(summary_cmp)

    # Vài mẫu CA-LA đúng, CA-base sai
    helped = both[both[HELP]]
    if len(helped) > 0:
        print(f"\n{len(helped)} mẫu CA-LA đúng mà CA-base sai:")
        display(helped.sample(min(10, len(helped)), random_state=0)[["label", "pred", "pred_la", "caption"]])
else:
    print("Chưa có predictions — chạy cell train trước.")

## 10. Dòng kết quả cho `reports/results/results.md`

Copy các dòng dưới vào `reports/results/results.md` rồi commit.

In [ ]:
for seed in SEEDS:
    for name in RUNS:
        path = OUT / run_name(name, seed) / "results_row.md"
        if path.exists():
            print(path.read_text(encoding="utf-8").strip().splitlines()[-1])

## 11. Hướng dẫn đọc kết quả

1. **Chỉ số xếp hạng:** `test macro-F1 (3 lớp)` (bỏ text-sarcasm) ổn định hơn 4 lớp do 7 mẫu text test gây nhiễu lớn (~0.15 F1 nếu sai/đúng thêm 1 mẫu).
2. **CA-LA vs CA-base:** Nếu LA tăng recall lớp hiếm (image/text-sarcasm) mà không làm sụt nặng not-sarcasm hoặc multi-sarcasm → kỹ thuật hiệu quả.
3. **CA-MX vs CA-base:** Mixup tạo ra smooth regularization; nếu val loss giảm chậm hơn nhưng ổn định hơn là tốt.
4. **CA-LAMX:** Kết hợp LA + Mixup có thể cạnh tranh hoặc tốt hơn từng kỹ thuật riêng lẻ, nhưng cũng có thể nhiễu loạn gradient — đọc F1 từng lớp cẩn thận.
5. **Calibration:** So sánh macro-F1 test trước/sau. Nếu không cải thiện, giữ kết quả train gốc.
6. **Hyperparameter:** Nếu CA-LA overfit (train loss < 0.3 mà val không tăng), thử `lr_head=3e-4` hoặc `weight_decay=0.1`.
7. **Không chốt model dựa vào private test** — private test chỉ chấm 1 lần theo giao thức chung.